# Установка моделей

In [ ]:
from pprint import pprint

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

# Установка токенизаторов

In [ ]:
torch.cuda.empty_cache()

In [ ]:
model_name_1 = "google/gemma-3-270m-it"
model_name_2 = "HuggingFaceTB/SmolLM2-360M-Instruct"
model_name_3 = "EleutherAI/pythia-410m"

tokenizer_1 = AutoTokenizer.from_pretrained(model_name_1)
model_1 = AutoModelForCausalLM.from_pretrained(model_name_1,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

tokenizer_2 = AutoTokenizer.from_pretrained(model_name_2)
model_2 = AutoModelForCausalLM.from_pretrained(model_name_2,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

tokenizer_3 = AutoTokenizer.from_pretrained(model_name_3)
model_3 = AutoModelForCausalLM.from_pretrained(model_name_3,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

# tokenizer_4 = AutoTokenizer.from_pretrained(model_name_4)
# model_4 = AutoModelForCausalLM.from_pretrained(model_name_4,
#                                                torch_dtype=torch.float16,
#                                                device_map="auto")

# Класс генерации forward pass'а

In [ ]:
from transformers.generation import GenerationMixin
from transformers.tokenization_utils_sentencepiece import SentencePieceBackend
from transformers.tokenization_utils_tokenizers import TokenizersBackend

TokenizerType = TokenizersBackend | SentencePieceBackend

class Generation:
    def __init__(
        self,
        llms: list[GenerationMixin],
        tokenizers: list[TokenizerType],
        top_k: int,
    ):
        self.llms = llms
        self.tokenizers = tokenizers
        self.chat_prefixes = []
        self.top_k = top_k
        self.device_1 = next(llms[0].parameters()).device
        self.device_2 = next(llms[0].parameters()).device

    @staticmethod
    def _build_chat_prefix(
        tokenizer: TokenizerType,
        user_message: str,
    ) -> str:
        messages = [
            {
                "role": "user",
                "content": user_message,
            }
        ]

        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    @staticmethod
    def _tokenize_prompt(
        tokenizer: TokenizerType,
        prompt: str,
        device: torch.device,
    ):
        return tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False,
        ).to(device)

    @staticmethod
    def _generate_log_probs(
        model: GenerationMixin,
        inputs,
    ) -> torch.Tensor:
        with torch.inference_mode():
            logits = model(**inputs).logits[:, -1, :]

        return torch.log_softmax(
            logits.float(),
            dim=-1,
        )

    def _get_top_distribution(
        self,
        log_probs: torch.Tensor,
        tokenizer: TokenizerType,
        model_number: int,
    ) -> dict[str, float]:
        k = min(self.top_k, log_probs.shape[-1])

        top_log_probs, top_token_ids = torch.topk(
            log_probs[0],
            k=k,
        )

        distribution: dict[str, float] = {}

        print(f"Модель {model_number}")

        for rank, (token_id, log_prob) in enumerate(
            zip(top_token_ids, top_log_probs, strict=True),
            start=1,
        ):
            token_id_int = token_id.item()

            token_text = tokenizer.decode(
                [token_id_int],
                skip_special_tokens=False,
            )

            probability = log_prob.exp().item()

            print(
                f"{rank}. "
                f"token={token_text!r}, "
                f"id={token_id_int}, "
                f"prob={probability:.6f}"
            )

            distribution[token_text] = (
                distribution.get(token_text, 0.0)
                + probability
            )

        return distribution

    def initialize_chat(self, user_message: str) -> None:
        for tokenizer in self.tokenizers:
            chat_prefix = self._build_chat_prefix(tokenizer=tokenizer,
                                                  user_message=user_message,)
            self.chat_prefixes.append(chat_prefix)

    def generate_pipe(
        self,
        generated_text: str,
        model_to_run: str = '',
    ):
        if self.chat_prefixes is None:
            raise RuntimeError(
                "Сначала вызови initialize_chat(user_message)"
            )

        print(f'Внутри generate pipe {model_to_run=}')
        distributions: dict[str, dict[str, float]] = {}
        if model_to_run == "":
            print('попали в условие запуска всех моделей')
            for idx, (llm, tokenizer) in enumerate(zip(self.llms, self.tokenizers, strict=True)):
                prompt = self.chat_prefixes[idx] + generated_text
                inputs = self._tokenize_prompt(
                    tokenizer=tokenizer,
                    prompt=prompt,
                    device=self.device_1,
                )

                log_probs = self._generate_log_probs(model=llm,
                                                     inputs=inputs)

                distributions[idx] = self._get_top_distribution(log_probs=log_probs,
                                                                tokenizer=tokenizer,
                                                                model_number=idx,)

        if model_to_run != "":
            print(f'{model_to_run=}')
            model_to_run = int(model_to_run)
            prompt = self.chat_prefixes[model_to_run] + generated_text
            print(f'попали в условие запуска модели {model_to_run}')
            print(f'{type(model_to_run)}')
            inputs = self._tokenize_prompt(tokenizer=self.tokenizers[model_to_run],
                                           prompt=prompt,
                                           device=self.device_1)

            log_probs = self._generate_log_probs(model=self.llms[model_to_run],
                                                 inputs=inputs,)

            distributions[model_to_run] = self._get_top_distribution(log_probs=log_probs,
                                                                     tokenizer=self.tokenizers[model_to_run],
                                                                     model_number=model_to_run,)

        return distributions

# Класс префиксной плотности

In [ ]:
import json

class PrefixDense:
    def __init__(self,
                 probs_generator: Generation,
                 input_str: str,
                 stop_token: str):
        self.prob_distribution_1 = {}
        self.prob_distribution_2 = {}
        self.prob_distributions = {}
        self.step_matrix = {}
        self.model_to_run = ''
        self.max_steps: int = 10
        self.probs_generator = probs_generator
        self.user_prompt = input_str
        self.generated_text = ""
        self.max_steps = 512
        self.stop_token = stop_token

    
    def runpipe(self):
        self.probs_generator.initialize_chat(
            user_message=self.user_prompt,
        )
        print(f'{self.max_steps}')
        for step in range(self.max_steps):
            print("###############################################################")
            print(f"Шаг: {step}")
            print("Пользовательский запрос:")
            print(self.user_prompt)
            print("Сгенерированное продолжение:")
            print(repr(self.generated_text))
            print("###############################################################")
            self.prob_distributions = self.probs_generator.generate_pipe(generated_text=self.generated_text)
            self.pretty_print(self.prob_distributions)
            ### Блок сопоставления токенов ###
            selected_text = self.match_chars()
            self.generated_text += selected_text
            print(f"Выбранный фрагмент: {selected_text!r}")
            if self.stop_token in selected_text:
                break
            print(f"Текущий ответ: {self.generated_text!r}")
        return self.generated_text

    @staticmethod
    def pretty_print(input_dict: dict, string: str = ''):
        print(f'{string}') 
        print(f'{json.dumps(input_dict, indent=2, ensure_ascii=False)}')

    def count_min_token_len_per_distrib(self, distribs: dict):
        distrib_min_len = {}
        distrib_max_len = {}
        for model_num, distrib in distribs.items():
            distrib_min_len[model_num] = min({len(key) for key in distrib})
            distrib_max_len[model_num] = max({len(key) for key in distrib})
        return distrib_min_len, distrib_max_len


    def find_the_suitest_token(self, 
                               distrib: dict, 
                               char_num: int, 
                               distrib_num: int):
        self.pretty_print(f'На вход find_the_suitest_token() подано распределение №{distrib_num}')
        self.pretty_print(distrib, 'Само распределение:')
        print(f"Расчет выполняется для индекса {char_num}")
        char_prob = {}
        new_distrib = {}
        for token, probability in distrib.items():
            if char_num >= len(token):
                print(f'Индекс {char_num} выходит за границы токена "{token}" с вероятностью {probability}')
                result = self.probs_generator.generate_pipe(generated_text=self.generated_text + token,
                                                            model_to_run=f"{distrib_num}")
                if isinstance(result, (tuple, list)):
                    result = result[distrib_num]
                self.pretty_print(result, f'Для токена "{token}" с вероятностью {probability} получили такое продолжение:')
                for token2, prob2 in result[distrib_num].items():
                    if not token2 or prob2 <= 0:
                        continue
                    ongoing_token = token + token2
                    ongoing_prob = probability*prob2
                    print(f'Полученный токен "{ongoing_token}", вероятность={ongoing_prob}')
                    if token not in new_distrib:
                        new_distrib[token] = {token2: ongoing_prob}
                    else:
                        new_distrib[token][token2] = ongoing_prob

                    if char_num < len(ongoing_token):
                        current_char = ongoing_token[char_num]
                        self.pretty_print(char_prob, 'На текущий момент')
                        if current_char not in char_prob:
                            char_prob[current_char] = ongoing_prob
                        else:
                            char_prob[current_char] += ongoing_prob
            else:
                current_char = token[char_num]
                char_prob[current_char] = (char_prob.get(current_char, 0.0) + probability)
                new_distrib[token] = (new_distrib.get(token, 0.0) + probability)
            self.pretty_print(char_prob,
                            f'На выходе из цикла подсчета вероятностей по {char_num}-му символу для модели №{distrib_num} получаем следующее:')

        if not char_prob:
            pass

        if new_distrib != distrib:
            future_new_distrib = {}
            for key, value in new_distrib.items():
                if isinstance(value, dict):
                    key_for_replacing = key
                    for key2, prob in value.items():
                        future_new_distrib[key+key2] = prob
            del new_distrib[key_for_replacing]
            new_distrib = {**new_distrib, **future_new_distrib}
            self.pretty_print(new_distrib, 'Полученный после комплита словарь, который подет на дальнейшую итерацию: ')    
        return new_distrib, char_prob
    
    def ensemble(self, ensemble_distr: dict):
        token_prob = {}
        for distrib in ensemble_distr.values():
            for token, prob in distrib.items():
                if token not in token_prob:
                    token_prob[token] = prob
                else:
                    token_prob[token] += prob
        
        the_most_popular_token = max(token_prob, key=token_prob.get)
        return the_most_popular_token

    def ensemble_str(self, ensemble_dict: dict, probs_list: list):
        ensembling_probs = {}
        for distr in ensemble_dict.values():
            self.pretty_print(distr, 'Распределение в цикле ensemble_str():')
            result = "".join(distr[key] for key in sorted(distr))
            print(f'Результат сложения префиксов: {result}')
            for model_num, distrib in probs_list.items():
                print(f'Для сложения токенов рассматривается распределение модели №{model_num}')
                for token, prob in distrib.items():
                    if token.startswith(result):
                        print(f'Токен {token} начинается с {result}, добавляем его вероятность')
                        if result not in ensembling_probs:
                            ensembling_probs[result] = prob
                        else:
                            ensembling_probs[result] += prob        
        pprint(f'{ensembling_probs=}')
        most_likely_token = max(ensembling_probs, key=ensembling_probs.get)
        return most_likely_token

   
    def match_chars(self):
        probs_list = self.prob_distributions
        distrib_min_len, distrib_max_len = self.count_min_token_len_per_distrib(distribs=probs_list)

        ensemble_dict = {}
        self.pretty_print(probs_list,'Распределение до входа в цикл обработки')
        self.pretty_print(distrib_min_len, 'Минимальная длина токена во всех моделях')
        self.pretty_print(distrib_max_len, 'Максимальная длина токена во всех моделях')
        prefix = ''
        max_char_steps = 128
        char_num = 0
        winning_char = {}
        while char_num < max_char_steps:
            print('####################################################################')
            print(f'Номер символа, по которому будет производится расчет: {char_num}')
            self.pretty_print(probs_list, 'Итерация производится по такому распределению:')
            for_suitest = {}
            dict_for_new_distrib = {}
            is_completion = False
            ### Блок определения локальных победителей ###
            for distrib_num, distrib in probs_list.items():
                new_distrib, char_dict = self.find_the_suitest_token(distrib=distrib,
                                                                     char_num=char_num,
                                                                     distrib_num=distrib_num) 
                self.pretty_print(new_distrib, 'После find_the_suitest_token()')
                for_suitest[distrib_num] = new_distrib
                self.pretty_print(new_distrib, 'Новое распределение (возможно, не изменившееся):')
                if distrib != new_distrib:
                    is_completion = True
                    print('Распределения отличаются. Значит, в new_distrib содержатся ключи продолжения токена. Формируем новое распределение для фильтрации')
                    dict_for_new_distrib[distrib_num] = new_distrib
                self.pretty_print(ensemble_dict, f'Словарь распределений символо на {char_num}-ом индексе ДО перезаписи: ')
                if char_num not in ensemble_dict:
                    ensemble_dict[char_num] = {distrib_num: char_dict}
                else:
                    ensemble_dict[char_num][distrib_num] = char_dict
                self.pretty_print(ensemble_dict, f'Словарь распределений символо на {char_num}-ом индексе ПОСЛЕ перезаписи: ')

            new_prob_list = {} 

            self.pretty_print(ensemble_dict, 'Топ символовов по вероятностям у моделей')
            prob_counter = {}
            repeat_counter = {}
            for distr in ensemble_dict[char_num].values():
                for token, prob in distr.items():
                    if token not in prob_counter:
                        prob_counter[token] = prob
                        repeat_counter[token] = 1
                    else:
                        prob_counter[token] += prob
                        repeat_counter[token] += 1

            self.pretty_print(prob_counter, 'Накопленная вероятность для определения победителя')
            self.pretty_print(repeat_counter, 'Словарь, отражающий число символов в распределениях моделей')

            recalculated_prob = {}
            for token, prob in prob_counter.items():
                num_count = repeat_counter[token]
                recalculated_prob[token] = prob/num_count

            self.pretty_print(recalculated_prob, 'Перерасчет средней вероятности с учетом повторений')

            winning_char[char_num] = max(recalculated_prob, key=recalculated_prob.get)
            print(f'"Победивший" символ — "{max(recalculated_prob, key=recalculated_prob.get)}"')
            self.pretty_print(winning_char, 'Словарь победивших символов')
            prefix = "".join(winning_char[key] for key in sorted(winning_char))
            print(f'Формируемый префикс, которому будем осуществлять фильтрацию токенов: "{prefix}"')


            ### Отбор только тех токенов, которые начинаются с prefix ###
            self.pretty_print(probs_list, 'Реальный слварь, по которому фильтруемся')
            if is_completion:
                self.pretty_print(dict_for_new_distrib, 'Потенциальный словарь для фильтра:')
                for distrib_num, distrib in for_suitest.items():
                    for token, prob in distrib.items():
                        if token.startswith(prefix):
                            if distrib_num not in new_prob_list:
                                new_prob_list[distrib_num] = {token: prob}
                            else:
                                if distrib_num not in new_prob_list:
                                    new_prob_list[distrib_num] = {token: prob}
                                else:
                                    new_prob_list[distrib_num][token] = prob
            else:
                for distrib_num, distrib in probs_list.items():
                    for token, prob in distrib.items():
                        if token.startswith(prefix):
                            if distrib_num not in new_prob_list:
                                new_prob_list[distrib_num] = {token: prob}
                            else:
                                if distrib_num not in new_prob_list:
                                    new_prob_list[distrib_num] = {token: prob}
                                else:
                                    new_prob_list[distrib_num][token] = prob
            ###############################################################
                            

            self.pretty_print(new_prob_list, f'Отфильтрованное распределение с теми токенами, которые начинаются на "{prefix}"')

            distrib_schema = []
            for model_num, distrib in new_prob_list.items():
                distrib_schema.append((model_num, distrib))


            print(f'Сформированная схема (номер_распределения, число_токенов_прошедших_фильтрацию):\n{distrib_schema=}')


            ones_counter = 0
            for elem in distrib_schema:
                if len(elem[1]) == 1:
                    ones_counter += 1

            if ones_counter == len(distrib_schema):
                print('В каждой из модели после фильтрации остался всего один токен, его и возвращаем')
                token_dict = distrib_schema[0][1]  # {'To': 0.8449...}
                token = next(iter(token_dict))     # 'To'
                print(f'Возвращаемое значение — {token}')
                return token

            char_num += 1
            probs_list = new_prob_list

# Инициализация пробной задачи

In [ ]:
task = "Lisa, Jack, and Tommy earned $60 from washing cars all week. However, half of the $60 was earned by Lisa. Tommy earned half of what Lisa earned. How much more money did Lisa earn than Tommy?"

# Тестирование Prefix-dense подхода

In [ ]:
agg = Generation(
    llms=[model_1,model_2,],
    tokenizers=[tokenizer_1,tokenizer_2],
    top_k=5,
)

agreement = PrefixDense(
    probs_generator=agg,
    input_str=task,
    stop_token='<|im_end|>'
)

answer = agreement.runpipe()

print("Итог:")
print(repr(answer))

# Тестирование модели 1

In [ ]:
# Qwen/Qwen2.5-0.5B-Instruct

streamer = TextStreamer(tokenizer_1, skip_prompt=True, skip_special_tokens=True)
prompt = task
input_ids = tokenizer_1.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
)["input_ids"].to(model_1.device)

output = model_1.generate(
    input_ids,
    do_sample=True,
    temperature=0.1,
    top_k=50,
    repetition_penalty=1.05,
    max_new_tokens=512,
    streamer=streamer,
)

# Тестирование модели 2

In [ ]:
streamer = TextStreamer(tokenizer_2, skip_prompt=True, skip_special_tokens=True)

prompt = task

input_ids = tokenizer_2.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer_2([input_ids], return_tensors="pt").to(model_2.device)

generated_ids = model_2.generate(
    **model_inputs,
    do_sample=True,
    temperature=0.1,
    top_k=50,
    repetition_penalty=1.05,
    max_new_tokens=512,
    # streamer=streamer,
)

generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]

response = tokenizer_2.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [ ]:
response

# Проверка на наборе данных

In [ ]:
import polars as pl

splits = {'train': 'main/train-00000-of-00001.parquet', 'test': 'main/test-00000-of-00001.parquet'}
df = pl.read_parquet("hf://datasets/openai/gsm8k/" + splits["train"])

In [ ]:
df = df.filter(pl.col('question').str.len_chars().is_between(135, 200))

In [ ]:
df

In [ ]:
questions = df['question'].to_list()
answers = df['answer'].to_list()

In [ ]:
from enum import Enum


class WhichModel(Enum):
    model_1: int = 1
    model_2: int = 2

class Solver:
    def __init__(self,
                 streamer: TextStreamer, 
                 models_list: list[GenerationMixin],
                #  model_2: GenerationMixin,
                #  model_3: GenerationMixin,
                 tokenizers_list: list[TokenizerType], 
                #  tokenizer_2: TokenizerType,
                #  tokenizer_3: TokenizerType,
                 topk: int):
        self.streamer = streamer
        self.models_list =models_list
        # self.model_2 = model_2
        # self.model_3 = model_3
        self.tokenizers_list = tokenizers_list
        # self.tokenizer_2 = tokenizer_2
        # self.tokenizer_3 = tokenizer_3
        self.topk = topk

    def solve_solo(self, which_model_generate: int, task: str):
        # if which_model_generate == WhichModel.model_1.value:
        model = self.models_list[which_model_generate]
        tokenizer = self.tokenizers_list[which_model_generate]
        # elif which_model_generate == WhichModel.model_2.value:
            # model = self.model_2
            # tokenizer = self.tokenizer_2
        input_ids = tokenizer.apply_chat_template([{"role": "user", "content": task}],
                                                    add_generation_prompt=True,
                                                    tokenize=False)

        model_inputs = tokenizer([input_ids], return_tensors="pt").to(model_1.device)
        
        generated_ids = model.generate(
                            **model_inputs,
                            do_sample=True,
                            temperature=0.1,
                            top_k=50,
                            repetition_penalty=1.05,
                            max_new_tokens=512,
                            # streamer=streamer,
                        )
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        return response

    def solve_ensemble(self, task: str):
        agg = Generation(llms=self.models_list,
                         tokenizers=self.tokenizers_list,
                         top_k=self.topk,)

        agreement = PrefixDense(
            probs_generator=agg,
            input_str=task,
            stop_token='<|im_end|>'
        )

        answer = agreement.runpipe()
        print("Итог:")
        print(repr(answer))
        return repr(answer)

In [ ]:
streamer = TextStreamer(tokenizer_2, skip_prompt=True, skip_special_tokens=True)
solver = Solver(streamer=streamer,
                models_list=[model_1, model_2, model_3],
                # model_2=model_2,
                # model_3=model_3,
                tokenizers_list=[tokenizer_1, tokenizer_2, tokenizer_3],
                # tokenizer_2=tokenizer_2,
                # tokenizer_3=tokenizer_3,
                topk=5)
delimiter = '#########################'
limit = 100

from pathlib import Path

output_dir = Path("Liquid350MATH_QWEN500M")  # можно указать любой путь, абсолютный или относительный
output_dir.mkdir(parents=True, exist_ok=True)  # создаст папку, если её нет

for question, answer in zip(questions[:limit], answers[:limit], strict=True):
    try:
        ensemble_result = solver.solve_ensemble(task=question)
        with open(output_dir / "ensemble_output.txt", "a", encoding="utf-8") as file:
            file.write(f"Right answer:\n{answer}\n")
            file.write(f"LLM's answer: {ensemble_result}\n")
            file.write(f"{delimiter}\n")
    except Exception as e:
        print(f'Возникла ошибка: {e}')
        continue

In [ ]:
import outlines
from enum import Enum

class Options(Enum):
    correct = 'correct'
    incorrect = 'incorrect'

class Checker:
    def __init__(self,
                 judge_llm: GenerationMixin,
                 tokenizer: TokenizerType,
                 list_of_files: list[str]):
        self.model = outlines.from_transformers(judge_llm, tokenizer)
        self.tokenizer = tokenizer
        self.list_of_files = list_of_files
        self.ready_for_check_dict = {}

    def get_dict_for_analysing(self):
        for file in self.list_of_files:
            with open(file, "r", encoding="utf-8") as file_read:
                content = file_read.read()
                tasks_solutions = content.split(delimiter)
                for elem in tasks_solutions:
                    right_answer = elem[elem.find('Right answer:\n')+len('Right answer:\n'):elem.find('####')]
                    llm_solution = elem[elem.find("LLM's answer: ")+len("LLM's answer: "):elem.find('<|im_end|>')]
                    if file not in self.ready_for_check_dict:
                        self.ready_for_check_dict[file] = []
                        self.ready_for_check_dict[file].append((right_answer, llm_solution))
                    else:
                        self.ready_for_check_dict[file].append((right_answer, llm_solution))

    def llm_check(self, system_prompt: str):
        # generator = outlines.generate.choice(self.model, ["valid", "invalid"])
        system_prompt = system_prompt

        for model_name, tasks in self.ready_for_check_dict.items():
            counter = 0
            model_counter = 0
            for task in tasks:
                print(f'Правильное решение: {task[0]}')
                print(f'Решение LLM: {task[1]}')
                counter += self.check_and_increment(right_answer=task[0],
                                                   llm_answer=task[1],
                                                   counter=counter,
                                                   )
                print('@@@@@@@@@@@@@@@@@@@@@@@@')
            print(f'Расчет числа правильно решенных задач завершен. Результат для {model_name} — {counter}/{len(tasks)}')

    def check_and_increment(self,
                            right_answer: str, 
                            llm_answer: str,
                            counter: int):
        input_text = f'Here is the correct answer to the problem:\n{right_answer}\n\
            Here is the solution to this problem from LLM:\n{llm_answer}\n\nIs this solution correct or incorrect? Compare only the final numbers'
        result = self.model(input_text, Options, max_new_tokens=10, temperature=0.5)
        print(f'Решение модели о правильности — {result}')
        if result == 'correct':
            return 1
        else:
            return 0

In [ ]:
model_name_3 = 'microsoft/Phi-4-mini-instruct'
tokenizer_3 = AutoTokenizer.from_pretrained(model_name_3)
model_3 = AutoModelForCausalLM.from_pretrained(model_name_3,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

In [ ]:
list_of_files = ['ensemble_output', 'model_1_output', 'model_2_output']
list_of_files = [f'{elem}.txt' for elem in list_of_files]

checker = Checker(judge_llm=model_3,
                  tokenizer=tokenizer_3,
                  list_of_files=list_of_files)
checker.get_dict_for_analysing()
checker.llm_check(system_prompt="You have been given the correct answer to the problem and a solution that needs to be checked for correctness. Determine whether the problem has been solved correctly")

In [ ]:
list_of_files = ['ensemble_output', 'model_1_output', 'model_2_output']
list_of_files = [f'{elem}.txt' for elem in list_of_files]

def get_dict_for_analysing(file_list: list[str]):
    ready_for_check = {}
    for file in file_list:
        with open(file, "r", encoding="utf-8") as file_read:
            content = file_read.read()
            tasks_solutions = content.split(delimiter)
            for elem in tasks_solutions:
                print(elem)
                print('!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!')
                right_answer = elem[elem.find('Right answer:\n')+len('Right answer:\n'):elem.find('####')]
                llm_solution = elem[elem.find("LLM's answer: ")+len("LLM's answer: "):elem.find('<|im_end|>')]
                if file not in ready_for_check:
                    ready_for_check[file] = []
                    ready_for_check[file].append((right_answer, llm_solution))
                else:
                    ready_for_check[file].append((right_answer, llm_solution))
    return ready_for_check

ready_for_check = get_dict_for_analysing(file_list=list_of_files)

In [ ]:
ready_for_check.keys()

In [ ]:
import outlines

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model = outlines.models.transformers(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

generator = outlines.generate.choice(model, ["valid", "invalid"])

counter = 0
system_prompt = 

def check_and_increment(input_text: str, 
                        counter, 
                        system_prompt="You have been given the correct answer to the problem and a solution that needs to be checked for correctness. Determine whether the problem has been solved correctly"):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": input_text}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    result = generator(prompt)
    
    if result == "valid":
        counter += 1
    
    return counter, result

# Пример использования
counter, verdict = check_and_increment("2 + 2 = 4", counter)
print(counter, verdict)  # 1 valid